In [1]:
import pandas as pd
import numpy as np
import os

# --- Configuration ---
FILE_PATH = 'data/records.csv' 
TARGET_COL = 'loan_default' # Adjust if named differently
X_COLS = ["account_age_months",
    "failed_txn_count_90d",
    "credit_utilisation_ratio",
    "avg_monthly_inflow_inr_thousands",
    "repayment_score",
    "irregular_salary_flag",
    "autopay_flag",
    "monthly_spend_inr_thousands"
] # X1 to X5 for Part 3

os.makedirs('outputs', exist_ok=True)

# Load Data
try:
    df = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    df = pd.read_excel(FILE_PATH)

# Standardise X1 to X5 over all 800 rows
mean_vec = df[X_COLS].mean().values
sd_vec = df[X_COLS].std(ddof=0).values # sample SD, ddof=1 if sample
X_std = (df[X_COLS].values - mean_vec) / sd_vec

# ==========================================
# Task 3.1: K-means from scratch
# ==========================================
print("--- Task 3.1: K-means ---")
def kmeans_pinned(X, K):
    n_samples = X.shape[0]
    # Starting centroids based on 0-based indices: 0, (n/K), 2*(n/K)...
    idx = [int(i * (n_samples / K)) for i in range(K)]
    centroids = X[idx].copy()
    
    max_iter = 1000
    for _ in range(max_iter):
        # Assign rows to nearest centroid (ties go to lower-numbered centroid)
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1) # argmin naturally breaks ties to lower index
        
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                new_centroids[k] = cluster_points.mean(axis=0)
            else:
                new_centroids[k] = centroids[k] # Handle empty cluster (rare)
                
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids

    # Calculate WCSS
    wcss = 0
    for i in range(n_samples):
        wcss += np.sum((X[i] - centroids[labels[i]])**2)
    return wcss, labels, centroids

wcss_results = {}
for K in [2, 3, 4, 5, 6]:
    wcss, _, _ = kmeans_pinned(X_std, K)
    wcss_results[K] = wcss
    print(f"wcss_k{K} = {wcss:.4f}")

--- Task 3.1: K-means ---
wcss_k2 = 4432.1510
wcss_k3 = 4146.8206
wcss_k4 = 3308.3234
wcss_k5 = 3040.4239
wcss_k6 = 2887.1647


In [3]:
# ==========================================
# Task 3.2: Choose K with the elbow
# ==========================================
print("\n--- Task 3.2: Elbow Method ---")
elbow_k = 5 # default
for k in [3, 4, 5]:
    if (k + 1)in wcss_results and (k+2)in wcss_results:
        drop_current = wcss_results[k] - wcss_results[k+1]
        drop_next = wcss_results[k+1] - wcss_results[k+2]
        if drop_current < 0.5 * drop_next:
            elbow_k = k
            break
print(f"elbow_k = {elbow_k}")
# Regardless of elbow_k calculated, the prompt says "use K = 3 for the rest of this Part"
print("Using K=3 for the rest of Part 3 as instructed.")



--- Task 3.2: Elbow Method ---
elbow_k = 5
Using K=3 for the rest of Part 3 as instructed.


In [5]:
# ==========================================
# Task 3.3: Profile the K=3 segments
# ==========================================
print("\n--- Task 3.3: Segments ---")
_, labels_3, _ = kmeans_pinned(X_std, 3)
df['segment_raw'] = labels_3

# Profile in original units
segment_profiles = df.groupby('segment_raw').agg(
    size=('account_age_months', 'size'),
    mean_X1=('account_age_months', 'mean'),
    mean_X2=('failed_txn_count_90d', 'mean'),
    mean_X3=('credit_utilisation_ratio', 'mean'),
    mean_X4=('avg_monthly_inflow_inr_thousands', 'mean'),
    mean_X5=('repayment_score', 'mean'),
    outcome_rate=(TARGET_COL, 'mean')
).reset_index()

# mean_value is the mean of the value column. Assuming 'value' is a column. 
# If not, adjust this to the actual column name.
if 'value' in df.columns:
    val_means = df.groupby('segment_raw')['value'].mean().values
    segment_profiles['mean_value'] = val_means
else:
    segment_profiles['mean_value'] = 0 # Fallback if 'value' column doesn't exist

# Order by ascending mean of X3 and rename to A, B, C
segment_profiles = segment_profiles.sort_values(by='mean_X3').reset_index(drop=True)
segment_profiles['segment'] = ['A', 'B', 'C']

# Save to CSV
segment_profiles[['segment', 'size', 'mean_X1', 'mean_X2', 'mean_X3', 'mean_X4', 'mean_X5', 'outcome_rate', 'mean_value']].to_csv('outputs/part3_segments.csv', index=False)
print("Saved outputs/part3_segments.csv")



--- Task 3.3: Segments ---
Saved outputs/part3_segments.csv


In [6]:
# ==========================================
# Task 3.4: Hierarchical clustering
# ==========================================
print("\n--- Task 3.4: Hierarchical Clustering ---")
# Distance matrix from the image
dist_matrix = np.array([
    [0.0, 7.9, 7.4, 2.9, 2.9, 6.1],
    [7.9, 0.0, 1.4, 5.9, 5.3, 2.0],
    [7.4, 1.4, 0.0, 5.8, 4.6, 1.3],
    [2.9, 5.9, 5.8, 0.0, 2.8, 4.5],
    [2.9, 5.3, 4.6, 2.8, 0.0, 3.4],
    [6.1, 2.0, 1.3, 4.5, 3.4, 0.0]
])
labels = ['E1', 'E2', 'E3', 'E4', 'E5', 'E6']

def agglomerative_clustering(dist_matrix, labels, linkage_type):
    n = len(labels)
    clusters = {i: [labels[i]] for i in range(n)}
    active = list(range(n))
    merge_history = []
    current_dist = dist_matrix.copy()
    
    merge_no = 1
    while len(active) > 1:
        # Find minimum distance
        min_dist = float('inf')
        pair_to_merge = None
        
        for i_idx in range(len(active)):
            for j_idx in range(i_idx + 1, len(active)):
                i, j = active[i_idx], active[j_idx]
                if current_dist[i, j] < min_dist:
                    min_dist = current_dist[i, j]
                    pair_to_merge = (i, j)
                    
        i, j = pair_to_merge
        c_a, c_b = clusters[i], clusters[j]
        
        # Sort members to ensure alphabetical order for the output string
        c_a_str = "+".join(sorted(c_a))
        c_b_str = "+".join(sorted(c_b))
        
        merge_history.append({
            'linkage': linkage_type, 'merge_no': merge_no, 
            'cluster_a': c_a_str, 'cluster_b': c_b_str, 'height': min_dist
        })
        
        # Create new cluster
        new_idx = n + merge_no - 1
        clusters[new_idx] = c_a + c_b
        
        # Calculate new distances
        for k_idx, k in enumerate(active):
            if k == i or k == j: continue
            
            if linkage_type == 'single':
                new_d = min(current_dist[i, k], current_dist[j, k])
            elif linkage_type == 'complete':
                new_d = max(current_dist[i, k], current_dist[j, k])
            elif linkage_type == 'average':
                new_d = (len(c_a) * current_dist[i, k] + len(c_b) * current_dist[j, k]) / (len(c_a) + len(c_b))
            
            # Expand matrix if needed
            if new_idx >= current_dist.shape[0]:
                pad = new_idx + 1 - current_dist.shape[0]
                current_dist = np.pad(current_dist, ((0, pad), (0, pad)), mode='constant', constant_values=np.inf)
                
            current_dist[new_idx, k] = new_d
            current_dist[k, new_idx] = new_d
            
        active.remove(i)
        active.remove(j)
        active.append(new_idx)
        merge_no += 1
        
    return pd.DataFrame(merge_history)

hier_dfs = []
for link in ['single', 'complete', 'average']:
    hier_dfs.append(agglomerative_clustering(dist_matrix, labels, link))

hier_df = pd.concat(hier_dfs)
hier_df.to_csv('outputs/part3_hierarchical.csv', index=False)
print("Saved outputs/part3_hierarchical.csv")



--- Task 3.4: Hierarchical Clustering ---
Saved outputs/part3_hierarchical.csv


In [8]:
# ==========================================
# Task 3.5: PCA by the seven steps
# ==========================================
print("\n--- Task 3.5: PCA ---")
# 1. Mean vector
print(f"PCA step 1 mean vector (standardised): {np.zeros(5)}")
# 2. Centre the data (Already done in X_std)
X_centered = X_std
# 3. Covariance matrix
cov_matrix = np.cov(X_centered, rowvar=False)
# 4. Eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
# Sort largest first
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]
# 5. Explained ratios
explained_ratios = eigenvalues / np.sum(eigenvalues)
cumulative_ratios = np.cumsum(explained_ratios)
components_for_80pct = np.argmax(cumulative_ratios >= 0.8) + 1
# 6. Keep top two eigenvectors
top2_vecs = eigenvectors[:, :2]
# 7. Sign convention: flip each eigenvector so entry of largest absolute value is positive
for i in range(2):
    if top2_vecs[np.argmax(np.abs(top2_vecs[:, i])), i] < 0:
        top2_vecs[:, i] *= -1

pc1_loadings = top2_vecs[:, 0]
pc2_loadings = top2_vecs[:, 1]

# Projections
projections = np.dot(X_centered, top2_vecs)

# ==========================================
# Generate part3_results.txt
# ==========================================
with open('outputs/part3_results.txt', 'w') as f:
    f.write("=== Task 3.1 ===\n")
    for k, v in wcss_results.items():
        f.write(f"wcss_k{k} = {v:.4f}\n")
    
    f.write("\n=== Task 3.2 ===\n")
    f.write(f"elbow_k = {elbow_k}\n")
    
    f.write("\n=== Task 3.3 ===\n")
    # Values are already in the CSV, just listing them here
    f.write("See outputs/part3_segments.csv for segment profiles.\n")
    f.write(f"size_segment_A = {segment_profiles.iloc[0]['size']}\n")
    f.write(f"size_segment_B = {segment_profiles.iloc[1]['size']}\n")
    f.write(f"size_segment_C = {segment_profiles.iloc[2]['size']}\n")
    
    f.write("\n=== Task 3.4 ===\n")
    # Clusters at cut (assuming 6 original points -> cut into 2 clusters)
    # I will extract clusters after 4 merges (6 - 4 = 2 clusters)
    def get_clusters_at_cut(df_link, num_merges):
        clusters = {i: [labels[i]] for i in range(6)}
        for i in range(num_merges):
            row = df_link.iloc[i]
            # find keys matching the string representation
            k_a = [k for k, v in clusters.items() if "+".join(sorted(v)) == row['cluster_a']][0]
            k_b = [k for k, v in clusters.items() if "+".join(sorted(v)) == row['cluster_b']][0]
            
            new_members = clusters[k_a] + clusters[k_b]
            del clusters[k_a]
            del clusters[k_b]
            clusters[max(clusters.keys(), default=-1)+1] = new_members
        return list(clusters.values())

    single_clusters = get_clusters_at_cut(hier_df[hier_df['linkage']=='single'].reset_index(drop=True), 4)
    complete_clusters = get_clusters_at_cut(hier_df[hier_df['linkage']=='complete'].reset_index(drop=True), 4)
    
    f.write(f"single_clusters_at_cut = {single_clusters}\n")
    f.write(f"complete_clusters_at_cut = {complete_clusters}\n")
    
    f.write("\n=== Task 3.5 ===\n")
    f.write(f"eigenvalue_sum = {np.sum(eigenvalues):.4f}\n")
    for i in range(5):
        f.write(f"eigenvalue_{i+1} = {eigenvalues[i]:.4f}\n")
    for i in range(5):
        f.write(f"explained_ratio_{i+1} = {explained_ratios[i]:.4f}\n")
    for i in range(5):
        f.write(f"cumulative_ratio_{i+1} = {cumulative_ratios[i]:.4f}\n")
    f.write(f"components_for_80pct = {components_for_80pct}\n")
    
    for i in range(5):
        f.write(f"pc1_loading_X{i+1} = {pc1_loadings[i]:.4f}\n")
    for i in range(5):
        f.write(f"pc2_loading_X{i+1} = {pc2_loadings[i]:.4f}\n")
        
    for i in range(3):
        f.write(f"row{i+1}_pc1 = {projections[i, 0]:.4f}\n")
        f.write(f"row{i+1}_pc2 = {projections[i, 1]:.4f}\n")
        
    # Segment means of PC1
    df['pc1'] = projections[:, 0]
    # Use labels from segment_raw
    pc1_means = df.groupby('segment_raw')['pc1'].mean()
    
    # We need to map segment_raw back to A, B, C based on the X3 ordering
    # segment_profiles is ordered A, B, C corresponding to the sorted segment_raw
    sorted_raw_labels = df.groupby('segment_raw')['credit_utilisation_ratio'].mean().sort_values().index.tolist()
    
    for idx, raw_lbl in enumerate(sorted_raw_labels):
        seg_name = ['A', 'B', 'C'][idx]
        f.write(f"mean_pc1_segment_{seg_name} = {pc1_means[raw_lbl]:.4f}\n")

print("Successfully wrote all outputs to the 'outputs/' directory.")


--- Task 3.5: PCA ---
PCA step 1 mean vector (standardised): [0. 0. 0. 0. 0.]
Successfully wrote all outputs to the 'outputs/' directory.
